# Single Error Decoding Test

Test if single physical errors cause logical errors in k=1 memory circuits.

Iterates over all single errors from the full DEM and decodes using the decomposed DEM with both PyMatching and Correlated PyMatching.

In [1]:
import stim
import numpy as np
import pymatching
from tqec import NoiseModel

In [2]:
# N/Z Circuit (k=1, non-compacted)
NZ_CIRCUIT_TEXT = """QUBIT_COORDS(0, 0) 0
QUBIT_COORDS(0, 2) 1
QUBIT_COORDS(0, 4) 2
QUBIT_COORDS(0, 6) 3
QUBIT_COORDS(1, 1) 4
QUBIT_COORDS(1, 3) 5
QUBIT_COORDS(1, 5) 6
QUBIT_COORDS(2, 0) 7
QUBIT_COORDS(2, 2) 8
QUBIT_COORDS(2, 4) 9
QUBIT_COORDS(2, 6) 10
QUBIT_COORDS(3, 1) 11
QUBIT_COORDS(3, 3) 12
QUBIT_COORDS(3, 5) 13
QUBIT_COORDS(4, 0) 14
QUBIT_COORDS(4, 2) 15
QUBIT_COORDS(4, 4) 16
QUBIT_COORDS(4, 6) 17
QUBIT_COORDS(5, 1) 18
QUBIT_COORDS(5, 3) 19
QUBIT_COORDS(5, 5) 20
QUBIT_COORDS(6, 0) 21
QUBIT_COORDS(6, 2) 22
QUBIT_COORDS(6, 4) 23
QUBIT_COORDS(6, 6) 24
RX 2 7 8 9 15 16 17 22
R 4 5 6 11 12 13 18 19 20
TICK
CZ 8 4 16 12 22 18
CX 9 5 15 11 17 13
TICK
CX 9 12 15 18 17 20
CZ 2 5
TICK
CX 7 4 9 6 15 12
CZ 8 5 16 13 22 19
TICK
CZ 8 11 16 19
TICK
CX 7 11 9 13 15 19
CZ 2 6 8 12 16 20
TICK
TICK
TICK
MX 2 7 8 9 15 16 17 22
DETECTOR(4, 4, 0) rec[-3]
DETECTOR(0, 4, 0) rec[-8]
DETECTOR(2, 2, 0) rec[-6]
DETECTOR(6, 2, 0) rec[-1]
SHIFT_COORDS(0, 0, 1)
TICK
RX 2 7 8 9 15 16 17 22
TICK
CZ 8 4 16 12 22 18
CX 9 5 15 11 17 13
TICK
CX 9 12 15 18 17 20
CZ 2 5
TICK
CX 7 4 9 6 15 12
CZ 8 5 16 13 22 19
TICK
CZ 8 11 16 19
TICK
CX 7 11 9 13 15 19
CZ 2 6 8 12 16 20
TICK
TICK
TICK
MX 2 7 8 9 15 16 17 22
DETECTOR(4, 2, 0) rec[-12] rec[-4]
DETECTOR(6, 2, 0) rec[-9] rec[-1]
DETECTOR(2, 2, 0) rec[-14] rec[-6]
DETECTOR(2, 0, 0) rec[-15] rec[-7]
DETECTOR(0, 4, 0) rec[-16] rec[-8]
DETECTOR(2, 4, 0) rec[-13] rec[-5]
DETECTOR(4, 4, 0) rec[-11] rec[-3]
DETECTOR(4, 6, 0) rec[-10] rec[-2]
SHIFT_COORDS(0, 0, 1)
TICK
RX 2 7 8 9 15 16 17 22
TICK
CZ 8 4 16 12 22 18
CX 9 5 15 11 17 13
TICK
CX 9 12 15 18 17 20
CZ 2 5
TICK
CX 7 4 9 6 15 12
CZ 8 5 16 13 22 19
TICK
CZ 8 11 16 19
TICK
CX 7 11 9 13 15 19
CZ 2 6 8 12 16 20
TICK
TICK
TICK
MX 2 7 8 9 15 16 17 22
M 4 5 6 11 12 13 18 19 20
DETECTOR(4, 4, 0) rec[-12] rec[-5] rec[-4] rec[-2] rec[-1]
DETECTOR(4, 2, 0) rec[-21] rec[-13]
DETECTOR(2, 2, 0) rec[-23] rec[-15]
DETECTOR(2, 0, 0) rec[-24] rec[-16]
DETECTOR(6, 2, 0) rec[-18] rec[-10]
DETECTOR(0, 4, 0) rec[-25] rec[-17]
DETECTOR(2, 4, 0) rec[-22] rec[-14]
DETECTOR(4, 4, 0) rec[-20] rec[-12]
DETECTOR(4, 6, 0) rec[-19] rec[-11]
DETECTOR(2, 2, 0) rec[-15] rec[-9] rec[-8] rec[-6] rec[-5]
DETECTOR(0, 4, 0) rec[-17] rec[-8] rec[-7]
DETECTOR(6, 2, 0) rec[-10] rec[-3] rec[-2]
OBSERVABLE_INCLUDE(0) rec[-8] rec[-5] rec[-2]
SHIFT_COORDS(0, 0, 1)"""

nz_circuit = stim.Circuit(NZ_CIRCUIT_TEXT)

In [3]:
# N/Z Circuit (k=1, compacted)
NZ_CIRCUIT_COMPACT_TEXT = """QUBIT_COORDS(0, 0) 0
QUBIT_COORDS(0, 2) 1
QUBIT_COORDS(0, 4) 2
QUBIT_COORDS(0, 6) 3
QUBIT_COORDS(1, 1) 4
QUBIT_COORDS(1, 3) 5
QUBIT_COORDS(1, 5) 6
QUBIT_COORDS(2, 0) 7
QUBIT_COORDS(2, 2) 8
QUBIT_COORDS(2, 4) 9
QUBIT_COORDS(2, 6) 10
QUBIT_COORDS(3, 1) 11
QUBIT_COORDS(3, 3) 12
QUBIT_COORDS(3, 5) 13
QUBIT_COORDS(4, 0) 14
QUBIT_COORDS(4, 2) 15
QUBIT_COORDS(4, 4) 16
QUBIT_COORDS(4, 6) 17
QUBIT_COORDS(5, 1) 18
QUBIT_COORDS(5, 3) 19
QUBIT_COORDS(5, 5) 20
QUBIT_COORDS(6, 0) 21
QUBIT_COORDS(6, 2) 22
QUBIT_COORDS(6, 4) 23
QUBIT_COORDS(6, 6) 24
RX 9 15 16 17 22
R 5 11 12 13 18
TICK
RX 2 8
R 4 19
CZ 16 12 22 18
CX 9 5 15 11 17 13
TICK
R 6 20
CZ 8 4 2 5 16 13 22 19
CX 9 12 15 18
TICK
CX 17 20 9 6 15 12
CZ 8 5 16 19
TICK
RX 7
CZ 8 11 16 20
CX 9 13 15 19
MX 22
TICK
CX 7 4
CZ 2 6 8 12
MX 9 15 16 17
RX 22
TICK
CX 7 11
MX 2 8
RX 9 15 16 17
CZ 22 18
TICK
RX 2 8
CZ 16 12 22 19
CX 9 5 15 11 17 13
TICK
CZ 8 4 2 5 16 13
CX 9 12 15 18 17 20
MX 22
TICK
MX 7 17
CX 9 6 15 12
CZ 8 5 16 19
RX 22
TICK
RX 7 17
CZ 8 11 16 20 22 18
CX 9 13 15 19
TICK
CX 7 4 17 13
CZ 2 6 8 12 22 19
MX 9 15 16
TICK
CX 7 11 17 20
MX 2 8 22
RX 9 15 16
TICK
MX 7 17
RX 2 8
CZ 16 12
CX 9 5 15 11
TICK
RX 7
CZ 8 4 2 5 16 13
CX 9 12 15 18
TICK
CX 7 4 9 6 15 12
CZ 8 5 16 19
M 18
TICK
CZ 8 11 2 6 16 20
CX 9 13 15 19
M 4 5
TICK
CX 7 11
CZ 8 12
MX 2 9 15 16
M 6 13 19 20
TICK
MX 7 8
M 11 12
DETECTOR(4, 4, 0) rec[-30]
DETECTOR(0, 4, 0) rec[-28]
DETECTOR(2, 2, 0) rec[-27]
DETECTOR(6, 2, 0) rec[-33]
DETECTOR(4, 2, 0) rec[-31] rec[-22]
DETECTOR(6, 2, 0) rec[-33] rec[-26]
DETECTOR(2, 2, 0) rec[-27] rec[-19]
DETECTOR(2, 0, 0) rec[-25] rec[-17]
DETECTOR(0, 4, 0) rec[-28] rec[-20]
DETECTOR(2, 4, 0) rec[-32] rec[-23]
DETECTOR(4, 4, 0) rec[-30] rec[-21]
DETECTOR(4, 6, 0) rec[-29] rec[-24]
DETECTOR(4, 4, 0) rec[-9] rec[-1] rec[-7] rec[-6] rec[-5]
DETECTOR(4, 2, 0) rec[-22] rec[-10]
DETECTOR(2, 2, 0) rec[-19] rec[-3]
DETECTOR(2, 0, 0) rec[-17] rec[-4]
DETECTOR(6, 2, 0) rec[-26] rec[-18]
DETECTOR(0, 4, 0) rec[-20] rec[-12]
DETECTOR(2, 4, 0) rec[-23] rec[-11]
DETECTOR(4, 4, 0) rec[-21] rec[-9]
DETECTOR(4, 6, 0) rec[-24] rec[-16]
DETECTOR(2, 2, 0) rec[-3] rec[-14] rec[-13] rec[-2] rec[-1]
DETECTOR(0, 4, 0) rec[-12] rec[-13] rec[-8]
DETECTOR(6, 2, 0) rec[-18] rec[-15] rec[-6]
OBSERVABLE_INCLUDE(0) rec[-13] rec[-1] rec[-6]
SHIFT_COORDS(0, 0, 1)
SHIFT_COORDS(0, 0, 1)
SHIFT_COORDS(0, 0, 1)"""

nz_circuit_compact = stim.Circuit(NZ_CIRCUIT_COMPACT_TEXT)

In [7]:
def get_detector_coordinates(circuit):
    """Extract detector coordinates from a circuit."""
    detector_coords = {}
    detector_idx = 0
    for inst in circuit.flattened():
        if inst.name == 'DETECTOR':
            coords = tuple(inst.gate_args_copy())
            detector_coords[detector_idx] = coords
            detector_idx += 1
    return detector_coords

def extract_detectors_and_observables(explained_err):
    """Extract detectors and observables from an ExplainedError."""
    error_detectors = []
    error_observables = []
    for dem_target_with_coords in explained_err.dem_error_terms:
        dem_target = dem_target_with_coords.dem_target
        if dem_target.is_relative_detector_id():
            error_detectors.append(dem_target.val)
        elif dem_target.is_logical_observable_id():
            error_observables.append(dem_target.val)
    return error_detectors, error_observables

def format_pauli_operator(flipped_pauli):
    """Format a flipped_pauli_product as a string like 'X0 Y1' or 'Z0'."""
    pauli_terms = []
    for target_with_coords in flipped_pauli:
        gate_target = target_with_coords.gate_target
        if hasattr(gate_target, 'value'):
            qubit_id = gate_target.value
            # Determine Pauli type (these are properties, not methods)
            if gate_target.is_x_target:
                pauli_type = 'X'
            elif gate_target.is_y_target:
                pauli_type = 'Y'
            elif gate_target.is_z_target:
                pauli_type = 'Z'
            else:
                pauli_type = '?'
            pauli_terms.append(f"{pauli_type}{qubit_id}")
    return ' '.join(sorted(pauli_terms))

def extract_qubit_information(explained_err):
    """Extract qubit IDs, coordinates, and Pauli operator from an ExplainedError, selecting the location with minimal qubits."""
    if not explained_err.circuit_error_locations:
        return [], [], ""
    
    # Find the circuit_error_location with minimal qubits
    min_qubits = float('inf')
    best_location = None
    best_coords_list = []
    best_pauli = ""
    
    for circuit_err_loc in explained_err.circuit_error_locations:
        qubits = set()
        coords_list = []
        flipped_pauli = circuit_err_loc.flipped_pauli_product
        for target_with_coords in flipped_pauli:
            gate_target = target_with_coords.gate_target
            if hasattr(gate_target, 'value'):
                qubit_id = gate_target.value
                qubits.add(qubit_id)
                if hasattr(target_with_coords, 'coords') and target_with_coords.coords:
                    coords_list.append((qubit_id, tuple(target_with_coords.coords)))
        
        num_qubits = len(qubits)
        if num_qubits < min_qubits:
            min_qubits = num_qubits
            best_location = sorted(qubits)
            best_coords_list = coords_list
            best_pauli = format_pauli_operator(flipped_pauli)
    
    return best_location if best_location else [], best_coords_list, best_pauli

def select_minimal_qubit_representatives(noisy_circuit, full_dem):
    """Get all error representatives.
    
    Each ExplainedError already represents a unique detector/observable pattern.
    circuit_error_locations contains all ways to produce that pattern.
    We'll select the minimal qubit location when extracting qubit information.
    """
    # Get all representatives (not reduced)
    # Each ExplainedError already represents a unique detector/observable pattern
    all_explained_errors = noisy_circuit.explain_detector_error_model_errors(
        dem_filter=full_dem,
        reduce_to_one_representative_error=False
    )
    
    return all_explained_errors, len(all_explained_errors)

In [8]:
def test_circuit(circuit, circuit_name, physical_error_rate=0.001):
    """Test a circuit by exhaustively checking all single physical errors."""
    print(f"\n{'='*60}")
    print(f"Testing {circuit_name} circuit (k=1)")
    print(f"{'='*60}")
    
    noise_model = NoiseModel.uniform_depolarizing(physical_error_rate)
    noisy_circuit = noise_model.noisy_circuit(circuit)
    
    # Create full DEM (not decomposed) - used as filter
    full_dem = noisy_circuit.detector_error_model(decompose_errors=False)
    
    # Create decomposed DEM (for decoding)
    decomposed_dem = noisy_circuit.detector_error_model(
        decompose_errors=True,
        ignore_decomposition_failures=False
    )
    
    print("Full DEM has", full_dem.num_detectors, "detectors,", full_dem.num_observables, "observables and", full_dem.num_errors, "errors")
    print("Decomposed DEM has", decomposed_dem.num_detectors, "detectors,", decomposed_dem.num_observables, "observables and", decomposed_dem.num_errors, "errors")
    
    # Get detector coordinates
    detector_coords = get_detector_coordinates(noisy_circuit)
    
    # Get explained errors and select minimal qubit representatives
    print(f"Getting error information...")
    explained_errors, total_representatives = select_minimal_qubit_representatives(noisy_circuit, full_dem)
    print(f"Found {total_representatives} error representatives")
    print(f"Selected {len(explained_errors)} unique error mechanisms (minimal qubit representatives)")
    
    # Create decoders
    decoders = {
        'PyMatching': pymatching.Matching.from_detector_error_model(
            decomposed_dem, enable_correlations=False
        ),
        'Correlated PyMatching': pymatching.Matching.from_detector_error_model(
            decomposed_dem, enable_correlations=True
        )
    }
    
    # Test each single error with all decoders
    print(f"\nTesting all {len(explained_errors)} single errors with both decoders...")
    all_results = {name: [] for name in decoders.keys()}
    
    for error_idx, explained_err in enumerate(explained_errors):
        # Extract detectors and observables
        error_detectors, error_observables = extract_detectors_and_observables(explained_err)
        
        # Extract qubit information (including Pauli operator)
        qubits, coords_list, pauli_operator = extract_qubit_information(explained_err)
        
        # Build detector pattern and actual observable
        detector_pattern = np.zeros(decomposed_dem.num_detectors, dtype=bool)
        actual_obs = np.zeros(circuit.num_observables, dtype=bool)
        
        for det in error_detectors:
            detector_pattern[det] = True
        for obs in error_observables:
            actual_obs[obs] = True
        
        # Test with each decoder
        det_samples = detector_pattern.reshape(1, -1)
        
        for decoder_name, matcher in decoders.items():
            enable_correlations = (decoder_name == 'Correlated PyMatching')
            predictions = matcher.decode_batch(det_samples, enable_correlations=enable_correlations)
            predicted_obs = predictions[0]
            
            # Check for logical errors
            if not np.array_equal(actual_obs, predicted_obs):
                all_results[decoder_name].append({
                    'error_id': error_idx,
                    'detectors': error_detectors,
                    'observables': error_observables,
                    'qubits': qubits,
                    'coords': coords_list,
                    'pauli_operator': pauli_operator,
                    'actual_obs': actual_obs.copy(),
                    'predicted_obs': predicted_obs.copy(),
                })
    
    # Report results for each decoder
    for decoder_name, single_errors in all_results.items():
        print(f"\nResults for {decoder_name}:")
        print(f"  Total single errors tested: {len(explained_errors)}")
        print(f"  Single errors causing logical errors: {len(single_errors)}")
        
        if len(single_errors) > 0:
            pct = 100 * len(single_errors) / len(explained_errors)
            print(f"  Percentage: {pct:.4f}%")
            print(f"\n  First 10 examples:")
            for i, err_info in enumerate(single_errors[:10], 1):
                print(f"    {i}. Error ID {err_info['error_id']}:")
                det_coords = [detector_coords.get(d, None) for d in err_info['detectors']]
                print(f"       Detector coordinates: {det_coords}")
                print(f"       Observables: {err_info['observables']}")
                if err_info['pauli_operator']:
                    print(f"       Pauli operator: {err_info['pauli_operator']}")
                if err_info['qubits']:
                    print(f"       Qubits: {err_info['qubits']}")
                if err_info['coords']:
                    print(f"       Qubit coordinates: {err_info['coords']}")
        else:
            print(f"  No single errors found that cause logical errors")

In [9]:
# Test N/Z circuit
test_circuit(nz_circuit, "N/Z (non-compacted)", physical_error_rate=0.001)


Testing N/Z (non-compacted) circuit (k=1)
Full DEM has 24 detectors, 1 observables and 219 errors
Decomposed DEM has 24 detectors, 1 observables and 287 errors
Getting error information...
Found 219 error representatives
Selected 219 unique error mechanisms (minimal qubit representatives)

Testing all 219 single errors with both decoders...

Results for PyMatching:
  Total single errors tested: 219
  Single errors causing logical errors: 0
  No single errors found that cause logical errors

Results for Correlated PyMatching:
  Total single errors tested: 219
  Single errors causing logical errors: 5
  Percentage: 2.2831%

  First 10 examples:
    1. Error ID 61:
       Detector coordinates: [(4.0, 2.0, 1.0), (4.0, 4.0, 1.0), (6.0, 2.0, 2.0), (4.0, 6.0, 2.0)]
       Observables: [0]
       Pauli operator: Y16 Y19
       Qubits: [16, 19]
       Qubit coordinates: [(16, (4.0, 4.0)), (19, (5.0, 3.0))]
    2. Error ID 71:
       Detector coordinates: [(4.0, 2.0, 1.0), (6.0, 2.0, 2.0), (4.0

In [10]:
# Test N/Z circuit (compacted)
test_circuit(nz_circuit_compact, "N/Z (compacted)", physical_error_rate=0.001)


Testing N/Z (compacted) circuit (k=1)
Full DEM has 24 detectors, 1 observables and 219 errors
Decomposed DEM has 24 detectors, 1 observables and 287 errors
Getting error information...
Found 219 error representatives
Selected 219 unique error mechanisms (minimal qubit representatives)

Testing all 219 single errors with both decoders...

Results for PyMatching:
  Total single errors tested: 219
  Single errors causing logical errors: 0
  No single errors found that cause logical errors

Results for Correlated PyMatching:
  Total single errors tested: 219
  Single errors causing logical errors: 7
  Percentage: 3.1963%

  First 10 examples:
    1. Error ID 22:
       Detector coordinates: [(2.0, 2.0, 0.0), (4.0, 2.0, 0.0), (2.0, 4.0, 0.0)]
       Observables: []
       Pauli operator: X11 Y8
       Qubits: [8, 11]
       Qubit coordinates: [(8, (2.0, 2.0)), (11, (3.0, 1.0))]
    2. Error ID 45:
       Detector coordinates: [(4.0, 2.0, 0.0), (2.0, 2.0, 0.0), (2.0, 4.0, 0.0)]
       Obser